# DataSense — Final Decision Test Notebook

**One notebook. Three tests. One decision.**

| Section | What it does | Why |
|---------|-------------|-----|
| **A. Setup** | Install RAPIDS + unsloth, load keys | One-time boot cost |
| **B. Load DataSense** | `unsloth/gemma-4-E2B-it` + LoRA adapter — exact same pattern as real working run | We need the real model |
| **C. Synthetic dataset** | Rich transactions dataset (default 1M rows, dial up to 20M) | Generic schema maps to any domain |
| **D. GPU sweep (no LLM)** | cuDF vs pandas, cuML vs sklearn across 100K–20M rows | Find where GPU wins are dramatic |
| **E. LLM-in-the-loop benchmark** | Model generates code for 4 tasks; pandas AND cuDF backends timed separately | The combined test that was missing |
| **F. Decision** | Ranked output + recommended problem statement | The output you need |

**Kaggle settings:** GPU T4 x2, Internet ON, Secret `E2B_API_KEY` (optional — Section E works without it)

**Estimated runtime:** ~40–60 min (RAPIDS install ~8 min, model load ~2 min, sweep ~20 min, LLM loop ~15 min)

---
## A. Setup

Run this cell first, wait for it to finish, then **Restart & Run All**.

In [ ]:
%%capture
# RAPIDS (cuDF + cuML)
!pip install -q --extra-index-url=https://pypi.nvidia.com \
    cudf-cu12 cuml-cu12 cupy-cuda12x

# DataSense model loader
!pip install -q -U unsloth
!pip install -q -U bitsandbytes accelerate peft transformers

# General
!pip install -q scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import os, re, time, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130

# ── RAPIDS ──────────────────────────────────────────────────────────────────
try:
    import cudf
    import cuml
    import cupy as cp
    RAPIDS_OK = True
    print(f"✓ RAPIDS  cuDF {cudf.__version__} | cuML {cuml.__version__}")
except Exception as e:
    RAPIDS_OK = False
    cudf = cuml = cp = None
    print(f"✗ RAPIDS import failed: {e}")
    print("  → Section D and E GPU benchmarks will be skipped.")

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU detected"
print("\nSetup done.")

---
## B. Load DataSense model

Same loading pattern as the real working Kaggle run:
- Base: `unsloth/gemma-4-E2B-it` (4-bit quantised via unsloth)
- LoRA adapter: `sanjaymalladi/DataSense-Modal-E2B-SFT`

In [ ]:
from unsloth import FastLanguageModel
import torch

BASE_MODEL     = "unsloth/gemma-4-E2B-it"
LORA_ADAPTER   = "sanjaymalladi/DataSense-Modal-E2B-SFT"
MAX_SEQ_LEN    = 4096
MAX_NEW_TOKENS = 1024

_t0 = time.perf_counter()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
_t_base = time.perf_counter()
model.load_adapter(LORA_ADAPTER)
_t_adapter = time.perf_counter()
FastLanguageModel.for_inference(model)
_t_ready = time.perf_counter()

print(f"Base model loaded  : {_t_base - _t0:.1f}s")
print(f"LoRA adapter loaded: {_t_adapter - _t_base:.1f}s")
print(f"Total model ready  : {_t_ready - _t0:.1f}s")


In [ ]:
# ── Generation helpers ────────────────────────────────────────────────────────
CODE_RE   = re.compile(r"```(?:python)?\s*\n(.*?)```", re.DOTALL)
ANSWER_RE = re.compile(r"\*\*Answer:\*\*\s*(.+)")

def extract_code(text):
    # Try finding complete code block
    hits = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if hits:
        return hits[-1].strip()
    # Fallback: if we have opening backticks but no closing backticks (truncated)
    hits_partial = re.findall(r"```(?:python)?\s*\n(.*)", text, re.DOTALL)
    if hits_partial:
        return hits_partial[-1].strip()
    return None

def extract_answer(text):
    m = ANSWER_RE.search(text)
    return m.group(1).strip() if m else None

def generate_timed(messages, max_new_tokens=MAX_NEW_TOKENS, temperature=0.2):
    """Run one model forward pass. Returns (text, tokenize_s, generate_s, n_new_tokens)."""
    _t = time.perf_counter()
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    tok_s = time.perf_counter() - _t

    _t = time.perf_counter()
    out = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen_s = time.perf_counter() - _t

    new_tokens = out[0][inputs.shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return text, tok_s, gen_s, len(new_tokens)

print("Model helpers ready.")


---
## C. Synthetic dataset

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
SWEEP_SIZES = [100_000, 1_000_000, 5_000_000, 20_000_000]  # Section D
N_ROWS      = 1_000_000                                      # Section E (LLM)

N_STORES, N_PRODUCTS, N_USERS, N_REGIONS = 250, 2000, 50_000, 8

def make_transactions(n, seed=42):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "txn_id"            : np.arange(n, dtype=np.int64),
        "date"              : (pd.Timestamp("2024-01-01")
                               + pd.to_timedelta(rng.integers(0, 365*24*3600, n), unit="s")
                              ).floor("D"),
        "store_id"          : rng.integers(0, N_STORES,   n, dtype=np.int32),
        "product_id"        : rng.integers(0, N_PRODUCTS, n, dtype=np.int32),
        "user_id"           : rng.integers(0, N_USERS,    n, dtype=np.int32),
        "region"            : rng.integers(0, N_REGIONS,  n, dtype=np.int8),
        "qty"               : rng.integers(1, 10,         n, dtype=np.int16),
        "price"             : rng.gamma(2.0, 20.0,        n).round(2),
        "discount_pct"      : rng.uniform(0, 0.35,        n).round(3),
        "return_flag"       : (rng.random(n) < 0.05).astype(np.int8),
        "support_tier"      : rng.integers(1, 5,          n, dtype=np.int8),
        "inventory"         : rng.integers(0, 5000,       n, dtype=np.int32),
        "days_since_restock": rng.integers(0, 60,         n, dtype=np.int16),
        "session_minutes"   : rng.gamma(3.0, 5.0,         n).round(2),
        "ticket_age_hours"  : rng.gamma(2.0, 8.0,         n).round(2),
        "sentiment"         : rng.normal(0, 1,             n).round(3),
        "feat_1"            : rng.normal(0, 1,             n).round(4),
        "feat_2"            : rng.normal(0, 1,             n).round(4),
        "feat_3"            : rng.normal(0, 1,             n).round(4),
        "feat_4"            : rng.normal(0, 1,             n).round(4),
    })
    df["revenue"]      = (df["qty"] * df["price"] * (1 - df["discount_pct"])).round(2)
    df["margin"]       = (df["revenue"] * rng.uniform(0.12, 0.42, n)).round(2)
    df["risk_score"]   = (
        0.45 * df["return_flag"].astype(float)
        + 0.20 * (df["ticket_age_hours"] / (df["ticket_age_hours"].max() + 1e-9))
        + 0.20 * (1 - np.clip(df["sentiment"], -2, 2) / 2)
        + 0.15 * (df["days_since_restock"] / (df["days_since_restock"].max() + 1e-9))
    )
    df["risk_label"]   = (df["risk_score"] > df["risk_score"].quantile(0.75)).astype(np.int8)
    df["target_flag"]  = df["risk_label"]
    df["target_value"] = df["risk_score"]
    return df

def load_bigquery_or_synthetic(n_rows):
    print("Attempting to connect to BigQuery for a public dataset...")
    try:
        from google.cloud import bigquery
        client = bigquery.Client()
        query = f"""
        SELECT 
            id as txn_id,
            user_id,
            product_id,
            created_at as date,
            sale_price as price,
            status
        FROM `bigquery-public-data.thelook_ecommerce.order_items`
        WHERE created_at IS NOT NULL
        LIMIT {n_rows}
        """
        print("Running BigQuery query on public thelook_ecommerce dataset...")
        df = client.query(query).to_dataframe()
        print(f"Successfully loaded {len(df):,} rows from BigQuery thelook_ecommerce dataset!")

        rng = np.random.default_rng(42)
        n = len(df)

        # ── Map real columns ──────────────────────────────────────────────────
        df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None).fillna(pd.Timestamp("2024-01-01"))
        df["price"] = df["price"].astype(float).fillna(50.0).round(2)
        
        # ── Generate synthetic columns for the rest ───────────────────────────
        df["store_id"]          = rng.integers(0, N_STORES, n, dtype=np.int32)
        df["region"]            = rng.integers(0, N_REGIONS, n, dtype=np.int8)
        df["qty"]               = rng.integers(1, 10, n, dtype=np.int16)
        df["discount_pct"]      = rng.uniform(0, 0.35, n).round(3)
        df["return_flag"]       = (df["status"] == "Returned").astype(np.int8)
        df["support_tier"]      = rng.integers(1, 5, n, dtype=np.int8)
        df["sentiment"]         = rng.normal(0, 1, n).round(3)
        df["ticket_age_hours"]  = rng.gamma(2.0, 8.0, n).round(2)
        df["days_since_restock"]= rng.integers(0, 60, n, dtype=np.int16)
        df["feat_1"]            = rng.normal(0, 1, n).round(4)
        df["feat_2"]            = rng.normal(0, 1, n).round(4)
        df["feat_3"]            = rng.normal(0, 1, n).round(4)
        df["feat_4"]            = rng.normal(0, 1, n).round(4)

        # ── Derived metrics ──────────────────────
        df["revenue"]     = (df["qty"] * df["price"] * (1 - df["discount_pct"])).round(2)
        df["margin"]      = (df["revenue"] * 0.30).round(2)
        
        df["risk_score"]   = (
            0.45 * df["return_flag"].astype(float)
            + 0.20 * (df["ticket_age_hours"] / (df["ticket_age_hours"].max() + 1e-9))
            + 0.20 * (1 - np.clip(df["sentiment"], -2, 2) / 2)
            + 0.15 * (df["days_since_restock"] / (df["days_since_restock"].max() + 1e-9))
        )
        df["risk_label"]  = (df["risk_score"] > df["risk_score"].quantile(0.75)).astype(np.int8)
        df["target_flag"] = df["risk_label"]
        df["target_value"]= df["risk_score"]

        # ── Select final columns (matching make_transactions schema) ──────────
        df = df[[
            "store_id", "product_id", "date", "region",
            "qty", "price", "discount_pct", "return_flag", "support_tier",
            "sentiment", "ticket_age_hours", "days_since_restock",
            "feat_1", "feat_2", "feat_3", "feat_4",
            "revenue", "margin", "risk_score", "risk_label",
            "target_flag", "target_value"
        ]].reset_index(drop=True)

        # Sanity check: drop rows with any remaining NaN
        before = len(df)
        df = df.dropna().reset_index(drop=True)
        if before - len(df) > 0:
            print(f"  Dropped {before - len(df):,} rows with NaN values.")
        print(f"  Final BigQuery df: {len(df):,} rows × {len(df.columns)} cols")
        return df

    except Exception as e:
        print(f"Could not load from BigQuery ({e}). Falling back to synthetic generator...")
        return make_transactions(n_rows)

print(f"Building/Fetching {N_ROWS:,}-row dataset...", end=" ")
pdf_llm = load_bigquery_or_synthetic(N_ROWS)
print(f"done. shape={pdf_llm.shape}")
pdf_llm.head(3)


---
## D. GPU acceleration sweep (no LLM)

Direct A/B benchmark — cuDF vs pandas, cuML vs sklearn — across row counts.  
**This chart decides your project direction.**

In [ ]:
if not RAPIDS_OK:
    print("RAPIDS not available — skipping Section D.")
else:
    from sklearn.linear_model import LinearRegression as skLinReg
    from sklearn.cluster import KMeans as skKMeans
    from sklearn.ensemble import RandomForestClassifier as skRFC
    from cuml.linear_model import LinearRegression as cuLinReg
    from cuml.cluster import KMeans as cuKMeans
    from cuml.ensemble import RandomForestClassifier as cuRFC

    FEATURES = ["feat_1", "feat_2", "feat_3", "feat_4", "price", "qty"]
    sweep_results = []

    def timeit(fn):
        t0 = time.perf_counter()
        fn()
        return time.perf_counter() - t0

    def record(op, n, engine, t):
        sweep_results.append({"op": op, "size": n, "engine": engine, "seconds": t})
        print(f"  [{engine:6s}] {op:22s}  n={n:>11,}  → {t:8.3f}s")

    for n in SWEEP_SIZES:
        print(f"\n{'='*60}\nn = {n:,}")
        pdf = make_transactions(n)
        gdf = cudf.from_pandas(pdf)

        # ── Data wrangling ────────────────────────────────────────────
        record("groupby_agg", n, "pandas",
               timeit(lambda: pdf.groupby(["store_id","product_id"]).agg({"revenue":"sum","qty":"sum"})))
        record("groupby_agg", n, "cudf",
               timeit(lambda: gdf.groupby(["store_id","product_id"]).agg({"revenue":"sum","qty":"sum"})))

        store_dim_pd = pd.DataFrame({"store_id": range(N_STORES),
                                     "store_tier": np.random.choice(["A","B","C"], N_STORES)})
        store_dim_gd = cudf.from_pandas(store_dim_pd)
        record("merge_join", n, "pandas",
               timeit(lambda: pdf.merge(store_dim_pd, on="store_id", how="left")))
        record("merge_join", n, "cudf",
               timeit(lambda: gdf.merge(store_dim_gd, on="store_id", how="left")))

        record("sort", n, "pandas", timeit(lambda: pdf.sort_values("revenue")))
        record("sort", n, "cudf",   timeit(lambda: gdf.sort_values("revenue")))

        record("rolling_window", n, "pandas",
               timeit(lambda: pdf.sort_values("date").groupby("store_id")["revenue"].rolling(7).mean()))
        try:
            record("rolling_window", n, "cudf",
                   timeit(lambda: gdf.sort_values("date").groupby("store_id")["revenue"].rolling(7).mean()))
        except Exception as e:
            print(f"  [cudf ] rolling_window error: {e}")

        # ── ML models ────────────────────────────────────────────────
        X_pd  = pdf[FEATURES]
        y_reg = pdf["target_value"]
        y_clf = pdf["target_flag"]
        X_gd  = gdf[FEATURES].astype("float32")
        y_rgd = gdf["target_value"].astype("float32")
        y_cgd = gdf["target_flag"].astype("int32")

        record("linreg_fit", n, "sklearn",
               timeit(lambda: skLinReg().fit(X_pd, y_reg)))
        record("linreg_fit", n, "cuml",
               timeit(lambda: cuLinReg().fit(X_gd, y_rgd)))

        record("kmeans_fit", n, "sklearn",
               timeit(lambda: skKMeans(n_clusters=8, n_init=3, random_state=0).fit(X_pd)))
        record("kmeans_fit", n, "cuml",
               timeit(lambda: cuKMeans(n_clusters=8, n_init=3, random_state=0).fit(X_gd)))

        if n <= 5_000_000:
            record("rf_classify_fit", n, "sklearn",
                   timeit(lambda: skRFC(n_estimators=50, max_depth=10, n_jobs=-1).fit(X_pd, y_clf)))
        else:
            print(f"  [sklearn] rf_classify_fit skipped at {n:,} — too slow (that IS the point)")
        record("rf_classify_fit", n, "cuml",
               timeit(lambda: cuRFC(n_estimators=50, max_depth=10).fit(X_gd, y_cgd)))

    print("\n✓ GPU sweep complete.")


In [ ]:
if not RAPIDS_OK:
    print("Skipping speedup computation — RAPIDS not available.")
else:
    res_df = pd.DataFrame(sweep_results)
    res_df.to_csv("gpu_sweep_results.csv", index=False)

    speedups = []
    for op in res_df["op"].unique():
        sub = res_df[res_df["op"] == op]
        cpu_engine = "pandas" if "pandas" in sub["engine"].values else "sklearn"
        gpu_engine = "cudf"   if "cudf"   in sub["engine"].values else "cuml"
        for n in sorted(sub["size"].unique()):
            c = sub[(sub["size"]==n) & (sub["engine"]==cpu_engine)]["seconds"]
            g = sub[(sub["size"]==n) & (sub["engine"]==gpu_engine)]["seconds"]
            if len(c) and len(g) and g.values[0] > 0:
                speedups.append({"op": op, "size": n,
                                  "speedup_x": c.values[0] / g.values[0],
                                  "cpu_s": c.values[0], "gpu_s": g.values[0]})

    speedup_df = pd.DataFrame(speedups)
    print("Speedup table (CPU time / GPU time):\n")
    display(speedup_df.pivot(index="size", columns="op", values="speedup_x").round(1))
    print("\nTop 10 speedups overall:")
    display(speedup_df.sort_values("speedup_x", ascending=False).head(10).round(2))

In [ ]:
if RAPIDS_OK:
    summary_rows = []
    for op in speedup_df["op"].unique():
        sub = speedup_df[speedup_df["op"] == op]
        summary_rows.append({
            "op"          : op,
            "max_speedup" : sub["speedup_x"].max(),
            "mean_speedup": sub["speedup_x"].mean(),
            "min_speedup" : sub["speedup_x"].min(),
            "best_size"   : int(sub.loc[sub["speedup_x"].idxmax(), "size"]),
        })
    summary_df = pd.DataFrame(summary_rows).sort_values("max_speedup", ascending=False)

    idea_map = {
        "rf_classify_fit" : "Risk / priority scoring (triage, fraud, ops priority)",
        "rolling_window"  : "Time-series alerting (freshness, anomaly tracking)",
        "merge_join"      : "Dashboard backend / enrichment joins",
        "groupby_agg"     : "Dashboard backend / BI acceleration",
        "linreg_fit"      : "Forecasting (demand, revenue, inventory)",
        "kmeans_fit"      : "Segmentation (weakest — avoid as core story)",
    }
    summary_df["project_direction"] = summary_df["op"].map(lambda x: idea_map.get(x, x))
    print("Project direction ranking by max GPU speedup:")
    display(summary_df[["project_direction","max_speedup","mean_speedup","min_speedup","best_size"]].round(1))

    fig, ax = plt.subplots(figsize=(11, 6))
    for op in speedup_df["op"].unique():
        sub = speedup_df[speedup_df["op"]==op].sort_values("size")
        ax.plot(sub["size"], sub["speedup_x"], marker="o", label=op)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.axhline(1, color="gray", linestyle="--", linewidth=1, label="1× (no gain)")
    ax.set_xlabel("rows")
    ax.set_ylabel("GPU speedup (CPU time / GPU time)")
    ax.set_title("Where does GPU acceleration actually pay off?")
    ax.legend()
    plt.tight_layout()
    plt.savefig("gpu_speedup_sweep.png", dpi=150)
    plt.show()
    print("Saved gpu_speedup_sweep.png")

---
## E. LLM-in-the-loop benchmark

**Key design:** the model is called **twice per task** — once with a pandas system prompt, once
with a cuDF system prompt that contains a full **few-shot cheat-sheet**.

Why two separate system prompts instead of a hint in the user message?
- The DataSense model was fine-tuned on pandas/sklearn code, not cuDF.
- Saying "use cuDF" without examples makes it guess at the API and often fail.
- The cuDF cheat-sheet embeds 8 concrete code patterns directly into the system prompt
  so the model has real examples to follow — no retraining needed.
- The pandas prompt stays lean so there is no interference between the two.

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class Task:
    name: str
    question: str
    category: str

TASKS: List[Task] = [
    Task(
        name="priority_ranking",
        question=(
            "Rank the top 25 rows by operational priority using return_flag, "
            "ticket_age_hours, days_since_restock, sentiment, and margin. "
            "Add a priority_score column and return the ranked dataframe."
        ),
        category="Risk / priority scoring",
    ),
    Task(
        name="rolling_alert",
        question=(
            "Compute a 7-day rolling average revenue per store. "
            "Flag the top 10 stores where today's revenue is more than 20% below "
            "the rolling average. Return a dataframe of those anomalies."
        ),
        category="Time-series alerting",
    ),
    Task(
        name="dashboard_summary",
        question=(
            "Create an aggregated dashboard summary grouped by region and support_tier. "
            "Show total revenue, average margin, return rate, average ticket_age_hours, "
            "and average sentiment. Return a compact summary table."
        ),
        category="Dashboard backend / BI",
    ),
    Task(
        name="risk_model",
        question=(
            "Train a fast classifier to predict risk_label from the numeric columns "
            "(feat_1 to feat_4, price, qty, discount_pct, ticket_age_hours, sentiment, "
            "days_since_restock). Return the top 20 highest-risk rows with their "
            "predicted probability as a new column called pred_risk."
        ),
        category="Risk scoring / classification",
    ),
]

print(f"{len(TASKS)} tasks defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# System prompts — one per backend
# The cuDF prompt embeds a full cheat-sheet so the model knows the exact API.
# The pandas prompt stays lean to avoid conflicting instructions.
# ════════════════════════════════════════════════════════════════════════════

CUDF_CHEATSHEET = """
# ════════════════════════════════════════════════════════════════
# cuDF / cuML CHEAT-SHEET  (few-shot examples — read carefully)
# ════════════════════════════════════════════════════════════════
#
# The variable `df` is ALREADY a cuDF DataFrame.
# Do NOT call cudf.from_pandas() or read any file.
#
# 1. groupby + agg  (identical syntax to pandas)
#    out = df.groupby(["store_id", "region"]).agg({"revenue": "sum", "qty": "mean"})
#
# 2. merge / join  (identical syntax to pandas)
#    out = df.merge(other_df, on="store_id", how="left")
#
# 3. sort  (identical syntax to pandas)
#    out = df.sort_values("revenue", ascending=False)
#
# 4. rolling window  (identical syntax to pandas)
#    out = df.sort_values("date").groupby("store_id")["revenue"].rolling(7).mean()
#
# 5. boolean filter  (identical syntax to pandas)
#    out = df[df["risk_score"] > 0.5]
#
# 6. priority score / ranking  (vectorized ONLY — DO NOT use df.apply or lambda row)
#    df["priority_score"] = (
#        df["return_flag"] * 0.45
#        + (df["ticket_age_hours"] / (df["ticket_age_hours"].max() + 1e-9)) * 0.20
#        + (1.0 - df["sentiment"].clip(-2, 2)/2) * 0.20
#        + (df["days_since_restock"] / (df["days_since_restock"].max() + 1e-9)) * 0.15
#    )
#    result = df.sort_values("priority_score", ascending=False).head(25)
#
# 7. cuML classifier  (sklearn-style API — use cuml, NOT sklearn)
#    import cuml
#    from cuml.ensemble import RandomForestClassifier as cuRFC
#    X = df[["feat_1", "feat_2", "feat_3", "feat_4"]].astype("float32")
#    y = df["risk_label"].astype("int32")
#    clf = cuRFC(n_estimators=50, max_depth=10)
#    clf.fit(X, y)
#    proba = clf.predict_proba(X)   # returns cuDF DataFrame/Series
#    # Use .iloc to access the column securely
#    df["pred_risk"] = proba.iloc[:, 1] if hasattr(proba, "iloc") else proba[:, 1]
#
# 8. convert result to pandas at the END only (for display)
#    result = df.head(20).to_pandas()
#
# CRITICAL RULES:
#   - The module name is `cudf` (all lowercase) — NOT `cuDF` or `CuDF`
#   - `cudf` is ALREADY imported — do NOT write `import cudf` at the top
#   - `df` is ALREADY a cuDF DataFrame — do NOT call cudf.from_pandas()
#   - .astype("float32") for ML feature columns
#   - .astype("int32")   for integer label columns
#   - DO NOT use: df.apply(lambda ...), df.iterrows(), df.itertuples()
#   - DO NOT import pandas — use cuDF methods only
#   - DO NOT import sklearn — use cuml equivalents only
# ════════════════════════════════════════════════════════════════
"""

SYSTEM_PROMPT_CUDF = (
    "You are DataSense, a GPU-accelerated data-science agent.\n"
    "You write cuDF + cuML code to answer questions about tabular datasets.\n"
    + CUDF_CHEATSHEET +
    "\nRESPONSE FORMAT:\n"
    "- Write exactly ONE ```python code block. No explanation outside it.\n"
    "- Assign the final answer to a variable named `result`.\n"
    "- Follow the cheat-sheet above exactly — no pandas, no sklearn."
).strip()

SYSTEM_PROMPT_PANDAS = (
    "You are DataSense, a data-science agent.\n"
    "You write pandas + sklearn code to answer questions about tabular datasets.\n"
    "The dataframe `df` is ALREADY a pandas DataFrame.\n"
    "NEVER use pd.read_csv(), pd.read_parquet(), open(), or any file I/O.\n"
    "ALL data is in the variable `df` — use it directly.\n"
    "\nCRITICAL RULES:\n"
    "1. Only use column names listed in the schema.\n"
    "2. Write vectorized operations — avoid row-by-row loops.\n"
    "\nRESPONSE FORMAT:\n"
    "- Write exactly ONE ```python code block. No explanation outside it.\n"
    "- Assign the final answer to a variable named `result`."
).strip()

def build_user_message(task, df):
    cols_info = "\n".join(f"  - {c}: {df[c].dtype}" for c in df.columns)
    sample    = df.head(2).to_string(index=False)
    return (
        f"Task:\n{task.question}\n\n"
        f"Dataset schema:\n- rows: {len(df):,}\n- columns:\n{cols_info}\n\n"
        f"Sample rows:\n{sample}"
    )

print(f"SYSTEM_PROMPT_PANDAS : {len(SYSTEM_PROMPT_PANDAS):,} chars")
print(f"SYSTEM_PROMPT_CUDF   : {len(SYSTEM_PROMPT_CUDF):,} chars (includes {len(CUDF_CHEATSHEET):,} chars cheat-sheet)")


In [ ]:
# ── Local executor (no E2B required) ─────────────────────────────────────────
# Runs generated code in a restricted namespace.
# cuDF backend: `df` is pre-converted to cuDF; cuml classes are injected.
# pandas backend: `df` is a pandas copy; sklearn classes are injected.

def run_code_locally(code, df_pandas, backend):
    """Returns (result_obj, elapsed_seconds, error_str_or_None)"""
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LogisticRegression, LinearRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score

    # Pre-inject cudf.pandas drop-in mode if cudf is requested
    if backend == "cudf" and RAPIDS_OK:
        try:
            import cudf.pandas
            cudf.pandas.install()
        except Exception as pi_err:
            print(f"  [warning] cudf.pandas.install failed: {pi_err}")

    # Patch file-reading functions so the model gets a helpful error
    def _no_file_read(*args, **kwargs):
        raise RuntimeError(
            "DO NOT read from a file. The dataframe is already loaded in `df`. "
            "Use `df` directly — no pd.read_csv(), open(), or any file I/O."
        )

    ns = {
        "__builtins__": __builtins__,
        "pd":   pd,
        "open": _no_file_read,  # prevent file I/O

        "np":   np,
        "result": None,
        "train_test_split":     train_test_split,
        "LogisticRegression":   LogisticRegression,
        "LinearRegression":     LinearRegression,
        "RandomForestClassifier": RandomForestClassifier,
        "roc_auc_score":        roc_auc_score,
    }

    if backend == "cudf" and RAPIDS_OK:
        import cuml
        from cuml.linear_model import LogisticRegression as cuLR
        from cuml.ensemble import RandomForestClassifier as cuRFC
        ns["cudf"] = cudf
        ns["cuDF"] = cudf   # alias for models that import cuDF (wrong case)
        ns["cuml"] = cuml
        ns["df"]   = cudf.from_pandas(df_pandas)
        ns["LogisticRegression"]    = cuLR
        ns["RandomForestClassifier"] = cuRFC
    else:
        ns["df"] = df_pandas.copy()

    t0 = time.perf_counter()
    try:
        exec(code, ns, ns)
        elapsed = time.perf_counter() - t0
        result  = ns.get("result")
        # Normalise cuDF → pandas for comparison
        if RAPIDS_OK and hasattr(result, "to_pandas") and not isinstance(result, pd.DataFrame):
            result = result.to_pandas()
        return result, elapsed, None
    except Exception as e:
        return None, time.perf_counter() - t0, f"{type(e).__name__}: {e}"

print("Local executor ready.")


In [ ]:
# ── Run the LLM benchmark ─────────────────────────────────────────────────────
# Per (task, backend) pair we:
#   1. Build the correct system prompt (SYSTEM_PROMPT_PANDAS or SYSTEM_PROMPT_CUDF)
#   2. Build a backend-agnostic user message (schema + task question)
#   3. Run a self-debugging execution loop of up to 8 iterations:
#      - Call generate_timed() → raw model output
#      - Extract the code block (including partial block recovery if truncated)
#      - Execute locally and record timing
#      - If error, feed error back to LLM to correct the code

BACKENDS    = ["pandas"] + (["cudf"] if RAPIDS_OK else [])
llm_results = []
final_dataframes = {}

for task in TASKS:
    print(f"\n{'='*65}\n{task.name}  [{task.category}]\n{'='*65}")
    for backend in BACKENDS:
        sys_prompt = SYSTEM_PROMPT_CUDF if backend == "cudf" else SYSTEM_PROMPT_PANDAS
        user_msg   = build_user_message(task, pdf_llm)

        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user",   "content": user_msg},
        ]

        MAX_ITER = 8
        attempt = 0
        exec_err = None
        code = None
        gen_s = 0.0
        n_tok = 0
        exec_s = None
        result = None

        while attempt < MAX_ITER:
            print(f"  Attempt {attempt + 1}/{MAX_ITER} for {backend.upper()}...")
            # LLM generation
            text_out, tok_s, gen_time, tokens = generate_timed(messages, max_new_tokens=MAX_NEW_TOKENS)
            gen_s += gen_time
            n_tok += tokens
            
            # Extract code
            code = extract_code(text_out)
            if not code:
                exec_err = "No python code block found in LLM output"
                print(f"    ⚠ Attempt {attempt + 1} failed: {exec_err}")
                messages.append({"role": "assistant", "content": text_out})
                messages.append({"role": "user", "content": "Please output the python code inside a single ```python ``` block."})
                attempt += 1
                continue
            
            # Try execution
            result, exec_s, exec_err = run_code_locally(code, pdf_llm, backend)
            if exec_err is None:
                # Succeeded!
                print(f"    ✓ Execution succeeded on attempt {attempt + 1} ({exec_s:.2f}s)")
                if result is not None and isinstance(result, pd.DataFrame):
                    print(f"    [Sample Output for {backend}]:")
                    display(result.head(3))
                    if backend == "cudf":
                        final_dataframes[task.name] = result
                break
            
            # Failed! Print error and append to messages for feedback
            print(f"    ✗ Execution error on attempt {attempt + 1}: {exec_err[:120]}")
            
            # Smart error-specific feedback
            if exec_err and ("FileNotFoundError" in exec_err or "No such file or directory" in exec_err):
                feedback = (
                    "Your code tried to read from a file. DO NOT use pd.read_csv() or any file I/O.\n"
                    "The dataframe is already loaded in the variable `df` — use it directly.\n"
                    "Output the corrected code inside exactly one ```python ``` block."
                )
            elif exec_err and ("ModuleNotFoundError" in exec_err and "cuDF" in exec_err):
                feedback = (
                    "The module `cuDF` does not exist. The correct name is `cudf` (all lowercase).\n"
                    "`cudf` is already in the namespace — do NOT import it. Use it directly.\n"
                    "Output the corrected code inside exactly one ```python ``` block."
                )
            elif exec_err and ("IntCastingNaNError" in exec_err or "non-finite" in exec_err):
                feedback = (
                    f"Your code failed: {exec_err[:120]}\n"
                    "The dataframe may have NaN or non-finite values. "
                    "Use .fillna(0) or .dropna() before casting to integer types.\n"
                    "Avoid df.astype(int8) on columns that may contain NaN.\n"
                    "Output the corrected code inside exactly one ```python ``` block."
                )
            elif exec_err and ("OverflowError" in exec_err or "out of bounds for int8" in exec_err):
                feedback = (
                    f"Your code failed: {exec_err[:120]}\n"
                    "An integer overflow occurred. Use int32 or int64 instead of int8 for large numeric columns.\n"
                    "Output the corrected code inside exactly one ```python ``` block."
                )
            else:
                feedback = (
                    f"Your generated code failed with the following execution error:\n"
                    f"```\n{exec_err}\n```\n\n"
                    f"Here was the code that failed:\n"
                    f"```python\n{code}\n```\n\n"
                    f"Please review the error message, correct your code, and output the corrected version "
                    f"inside exactly one ```python ``` block."
                )
            messages.append({"role": "assistant", "content": text_out})
            messages.append({"role": "user", "content": feedback})
            attempt += 1

        status = "OK" if exec_err is None else f"ERROR: {exec_err[:120]}"
        print(f"Final result [{backend}]: {exec_s:.2f}s if exec_s else 0.0s  →  {status}")

        llm_results.append({
            "task"        : task.name,
            "category"    : task.category,
            "backend"     : backend,
            "generation_s": gen_s,
            "exec_s"      : exec_s,
            "total_s"     : gen_s + (exec_s or 0.0),
            "n_tokens"    : n_tok,
            "exec_ok"     : exec_err is None,
            "error"       : exec_err,
            "code"        : code,
        })

print("\n✓ LLM benchmark complete.")


In [ ]:
# ── Results table ────────────────────────────────────────────────────────────
bench_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "code"}
    for r in llm_results
])
bench_df.to_csv("datasense_llm_benchmark.csv", index=False)
display(bench_df[["task","backend","generation_s","exec_s","total_s","n_tokens","exec_ok","error"]].round(2))

# ── Speedup per task ─────────────────────────────────────────────────────────
llm_speedup_rows = []
for task in TASKS:
    sub  = bench_df[bench_df["task"] == task.name]
    p_ok = sub[(sub["backend"]=="pandas") & sub["exec_ok"]]
    g_ok = sub[(sub["backend"]=="cudf")   & sub["exec_ok"]]
    if not p_ok.empty and not g_ok.empty:
        p, g = p_ok.iloc[0], g_ok.iloc[0]
        llm_speedup_rows.append({
            "task"           : task.name,
            "category"       : task.category,
            "exec_speedup_x" : p["exec_s"] / g["exec_s"]   if g["exec_s"]   else None,
            "total_speedup_x": p["total_s"] / g["total_s"] if g["total_s"] else None,
            "pandas_exec_s"  : p["exec_s"],
            "cudf_exec_s"    : g["exec_s"],
        })

if llm_speedup_rows:
    llm_speedup_df = pd.DataFrame(llm_speedup_rows)
    print("\nLLM task execution speedups (pandas exec / cuDF exec):")
    display(llm_speedup_df.round(2))
else:
    print("Not enough successful runs on both backends to compute speedups.")
    llm_speedup_df = pd.DataFrame()

In [ ]:
# ── Visualise LLM benchmark ──────────────────────────────────────────────────
if not bench_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    for backend in bench_df["backend"].unique():
        sub = bench_df[bench_df["backend"]==backend]
        ax.plot(sub["task"], sub["total_s"], marker="o", label=backend)
    ax.set_ylabel("total latency (s) — gen + exec")
    ax.set_title("End-to-end LLM task latency\n(gen ≈ equal; exec gap shows GPU win)")
    ax.legend()
    plt.sca(ax); plt.xticks(rotation=20, ha="right")

    ax = axes[1]
    if not llm_speedup_df.empty and "exec_speedup_x" in llm_speedup_df.columns:
        colors = ["#00aa44" if v >= 5 else "#ddaa00" if v >= 2 else "#cc3333"
                  for v in llm_speedup_df["exec_speedup_x"].fillna(0)]
        ax.bar(llm_speedup_df["task"], llm_speedup_df["exec_speedup_x"], color=colors)
        ax.axhline(1, color="gray", linestyle="--")
        ax.set_ylabel("exec speedup (pandas / cuDF)")
        ax.set_title("Execution speedup per task\n(green ≥5× | yellow ≥2× | red <2×)")
        plt.sca(ax); plt.xticks(rotation=20, ha="right")
    else:
        ax.text(0.5, 0.5, "Not enough data", ha="center", va="center",
                transform=ax.transAxes)

    plt.tight_layout()
    plt.savefig("datasense_llm_benchmark.png", dpi=150)
    plt.show()

---
## F. Final decision

In [ ]:
print("=" * 65)
print("FINAL DECISION SUMMARY")
print("=" * 65)

if RAPIDS_OK and "summary_df" in dir():
    sweep_winner = summary_df.iloc[0]
    print(f"""
GPU Sweep Winner (Section D):
  Operation   : {sweep_winner['op']}
  Max speedup : {sweep_winner['max_speedup']:.1f}×
  Mean speedup: {sweep_winner['mean_speedup']:.1f}×
  Direction   : {sweep_winner['project_direction']}
""")

if not llm_speedup_df.empty:
    llm_winner = llm_speedup_df.sort_values("exec_speedup_x", ascending=False).iloc[0]
    print(f"""
LLM-in-loop Winner (Section E):
  Task        : {llm_winner['task']}
  Exec speedup: {llm_winner['exec_speedup_x']:.1f}×
  Category    : {llm_winner['category']}
""")

DECISION_GUIDE = """
Decision rule (use the bigger of the two signals):

  rf_classify_fit wins (>20×)  →  Build: RISK / PRIORITY SCORING TOOL
     User:   Ops manager, support lead, fraud analyst
     Output: Ranked rows with predicted risk probability
     GPU story: cuML RF trains 30× faster → refresh scores every 5 min
                instead of daily → real-time triage becomes possible

  rolling_window wins (>20×)   →  Build: TIME-SERIES ALERTING TOOL
     User:   Store manager, site-reliability engineer
     Output: Alert list — stores with anomalous rolling metrics
     GPU story: 69× cuDF rolling → dashboard refreshes in <1s vs 60s

  merge_join / groupby win      →  Build: GPU-ACCELERATED DASHBOARD BACKEND
     User:   Analyst using Looker / BI tool
     Output: Aggregated KPI table (revenue, margin, ticket age by region)
     GPU story: 40× join speed → BigQuery-scale queries on-GPU

  Nothing >5× consistently     →  Pivot: BigQuery + Gemini NL interface
     Don't force the GPU story; lean on Google Cloud data-layer instead
"""
print(DECISION_GUIDE)

with open("final_decision_notes.txt", "w") as f:
    f.write("DataSense hackathon — final decision notes\n")
    if RAPIDS_OK and "summary_df" in dir():
        f.write("\nGPU sweep ranking:\n")
        f.write(summary_df[["project_direction","max_speedup","mean_speedup"]].to_string(index=False))
    if not llm_speedup_df.empty:
        f.write("\n\nLLM task speedups:\n")
        f.write(llm_speedup_df.to_string(index=False))
    f.write(DECISION_GUIDE)

print("\nSaved:")
print("  gpu_sweep_results.csv")
print("  gpu_speedup_sweep.png")
print("  datasense_llm_benchmark.csv")
print("  datasense_llm_benchmark.png")
print("  final_decision_notes.txt")

## G. Final Output Showcase & Export (Problem Statement Requirement)
The problem statement explicitly requires a **useful output** (alert, ranking, risk score). 
Here we export the final decisions generated by the GPU-accelerated LLM pipeline so they can be consumed by a dashboard or downstream ops team.

In [ ]:
import os

print("=" * 65)
print("FINAL DECISION OUTPUTS")
print("=" * 65)

if 'risk_model' in final_dataframes:
    print("\n🚀 1. SUPPORT LEAD: HIGH RISK RETURN RANKING")
    risk_df = final_dataframes['risk_model']
    display(risk_df.head(10))
    risk_df.to_csv("high_risk_returns.csv", index=False)
    print("Saved -> high_risk_returns.csv (Ready for dashboarding)")

if 'rolling_alert' in final_dataframes:
    print("\n⚠️ 2. OPS MANAGER: STORE REVENUE ANOMALIES")
    alert_df = final_dataframes['rolling_alert']
    display(alert_df.head(10))
    alert_df.to_csv("revenue_anomalies.csv", index=False)
    print("Saved -> revenue_anomalies.csv (Ready for alerts)")

print("\n✓ Pipeline complete: Ingest (BigQuery) -> Analyze (LLM) -> Model (cuML GPU) -> Output (CSV/Dashboard)")